In [ ]:


from __future__ import annotations

from dataclasses import dataclass, field
from typing import List


@dataclass
class CommunityConfig:
    """
    All hyperparameters for the community detection pipeline.

    Change here, never hardcode in detection / scoring / tracking modules.

    Graph weighting
    ---------------
    alpha : float
        Monetary continuity penalty — exp(-alpha * |log(a_src / a_dst)|).
        Higher = stricter capital preservation requirement.
    beta : float
        Temporal decay penalty — exp(-beta * avg_gap / delta_w).
        Higher = edges further apart in time get lower weight.
    weight_filter_thresh : float
        Drop second-order edges with final weight below this value.
    top_k_neighbors : int
        Keep at most this many outgoing neighbours per node after weighting.
        0 = disabled (keep all).

    Community detection
    -------------------
    method : str
        Primary algorithm. Options: "leiden", "infomap".
        "leiden"  — directed modularity, good for structural communities.
        "infomap" — flow-based, good for detecting money-trapping patterns.
    min_comm_size : int
        Discard communities smaller than this. Prevents trivial singletons.
    resolution : float
        Leiden resolution parameter. Higher = more, smaller communities.
    lambda_temporal : float
        Temporal regularisation strength (λ).
        Penalty for label changes between consecutive windows.
        0 = no regularisation; 0.5 = moderate stability pressure.

    Recursive split
    ---------------
    s_max : int
        Communities larger than this are recursively split.
    resolution_multiplier : float
        Multiply resolution by this at each recursion level.
    max_recursion_depth : int
        Hard stop on recursion depth.

    Role classification
    -------------------
    role_threshold : float
        |net_flow_ratio| > threshold → node classified as source (>0) or sink (<0).
    layering_consistency : float
        flow_consistency > threshold → node classified as layering intermediary.

    Temporal tracking
    -----------------
    window_sizes : list[int]
        Rolling-window sizes in steps, aligned with GraphConfig.
    stride : int
        Step increment between windows.
    jaccard_thresh : float
        Minimum Jaccard overlap to consider two communities the same entity
        across consecutive windows.
    tracking_memory : int
        Number of past windows to match against during tracking.

    Suspicion scoring
    -----------------
    Weights for the composite suspicion score S(C):
        S(C) = w_internal_flow  * InternalFlow
             + w_reciprocity    * Reciprocity
             + w_persistence    * Persistence
             + w_motif          * MotifEnrichment
             - w_external_noise * ExternalNoise
    All weights should sum to approximately 1.0.

    alert_thresh : float
        Flag community if SAR-edge ratio exceeds this value.
    min_flow_for_scoring : float
        Skip scoring communities with total internal flow below this.
    global_susp_threshold : float
        Communities with S(C) above this are added to the investigation shortlist.

    size_penalty_gamma : float
        Log size-penalty coefficient in the suspicion score.
        Reduces the score of large communities to prevent size-bias.
        S_penalised = S - gamma * log(1 + |C|).
        Configurable here so experiments can turn it off (set to 0.0).

    Output
    ------
    top_k_export : int
        Number of highest-scoring communities to include in the export.
    """

    # ── Graph weighting ───────────────────────────────────────────────────────
    alpha: float = 1.0
    beta: float = 0.5
    weight_filter_thresh: float = 0.01
    top_k_neighbors: int = 10

    # ── Community detection ───────────────────────────────────────────────────
    method: str = "leiden"          # "leiden" or "infomap"
    min_comm_size: int = 3
    resolution: float = 1.0
    lambda_temporal: float = 0.3    # temporal regularisation strength

    # ── Recursive split ───────────────────────────────────────────────────────
    s_max: int = 50
    resolution_multiplier: float = 1.5
    max_recursion_depth: int = 3

    # ── Role classification ───────────────────────────────────────────────────
    role_threshold: float = 0.3
    layering_consistency: float = 0.8

    # ── Temporal tracking ─────────────────────────────────────────────────────
    window_sizes: List[int] = field(default_factory=lambda: [7, 14, 30])
    stride: int = 7
    jaccard_thresh: float = 0.25
    tracking_memory: int = 3

    # ── Suspicion scoring weights ─────────────────────────────────────────────
    w_internal_flow: float = 0.30
    w_reciprocity: float = 0.20
    w_persistence: float = 0.25
    w_motif: float = 0.15
    w_external_noise: float = 0.10

    alert_thresh: float = 0.30
    min_flow_for_scoring: float = 10_000.0
    global_susp_threshold: float = 0.45

    # ── Output ────────────────────────────────────────────────────────────────
    top_k_export: int = 50
    size_penalty_gamma: float = 0.05   # log size-penalty coefficient in score_communities


__all__ = ["CommunityConfig"]

In [ ]:
"""
 Community detection on directed weighted window graphs.

Three public functions:

    detect_communities(wg, cfg)
        Primary entry point. Takes a WindowGraph from weighting.py.
        Returns Dict[node_id, community_id] for every active node.
        Method is selected by cfg.method: "leiden" or "infomap".

    compute_node_roles(window_df, cfg)
        Classify each node as: source / sink / layering / neutral.
        Input is the raw transaction DataFrame for the window.

    build_relay_edges(snapshot_edges)
        Preserve relay structure for edge-level analysis.
        Returns an aggregated directed edge table.

Algorithm notes (community_detection.md §1–6):
    - Primary: directed modularity via igraph Leiden (CPU).
      igraph supports directed=True in its Leiden implementation.
    - Fallback: if igraph unavailable, use networkx greedy_modularity.
    - GPU path (cuGraph): attempted if RAPIDS is present and cfg allows it.
    - Infomap: use infomap package if cfg.method == "infomap".
    - Symmetrization: only if the algorithm explicitly requires undirected.
      Any symmetrization must be documented in the call site.

Guarantees:
    - Time      : operates on a single WindowGraph (no cross-window state).
    - Direction : A[i,j] used directly; d_out / d_in kept separate.
                  Symmetrization is never applied silently.
    - Memory    : igraph graph freed after each call; no cuDF/cuGraph required.
"""

from __future__ import annotations

import gc
from typing import Dict, List

import numpy as np
import pandas as pd




# ---------------------------------------------------------------------------
# Algorithm availability checks (lazy imports — no crash at module load)
# ---------------------------------------------------------------------------

def _try_igraph():
    try:
        import igraph as ig
        return ig
    except ImportError:
        return None


def _try_infomap():
    try:
        import infomap as im
        return im
    except ImportError:
        return None


def _try_cugraph():
    try:
        import cugraph
        import cudf
        return cugraph, cudf
    except ImportError:
        return None, None


# ---------------------------------------------------------------------------
# Primary entry point
# ---------------------------------------------------------------------------

def detect_communities(
    wg: WindowGraph,
    cfg: CommunityConfig,
) -> Dict[int, int]:
    """
    Detect communities in a single time-window directed weighted graph.

    Parameters
    ----------
    wg : WindowGraph
        Output of build_window_graph(). Contains A (CSR), d_out, d_in, m_t.
    cfg : CommunityConfig
        method, resolution, min_comm_size, s_max, etc.

    Returns
    -------
    Dict[node_id, community_id]
        Community assignment for every node present in the window.
        Nodes not in any edge are assigned community -1.

    Notes
    -----
    - Method priority: igraph Leiden → igraph community_multilevel →
      networkx greedy_modularity → trivial (all nodes in one community).
    - "infomap" method uses the infomap package if available; falls back
      to the default Leiden path if not installed.
    - cuGraph is attempted only when method == "leiden" and RAPIDS is
      available; symmetrization is then applied and documented.
    """
    if wg.A.nnz == 0:
        return {}

    if cfg.method == "infomap":
        labels = _run_infomap(wg, cfg)
    else:
        labels = _run_leiden(wg, cfg)

    # Apply minimum community size filter
    if cfg.min_comm_size > 1:
        labels = _filter_small_communities(labels, cfg.min_comm_size)

    return labels


# ---------------------------------------------------------------------------
# Leiden path
# ---------------------------------------------------------------------------

def _run_leiden(wg: WindowGraph, cfg: CommunityConfig) -> Dict[int, int]:
    """
    Run Leiden community detection.

    Tries, in order:
        1. igraph Leiden with directed=True on the directed graph.
        2. igraph community_multilevel (undirected fallback — symmetrised).
        3. networkx greedy_modularity (undirected fallback).
        4. Trivial: all nodes in community 0.
    """
    ig = _try_igraph()
    if ig is not None:
        return _leiden_igraph(wg, cfg, ig)

    # networkx fallback
    try:
        import networkx as nx
        return _leiden_networkx(wg, cfg, nx)
    except ImportError:
        pass

    # Trivial: every node is its own community
    active_nodes = _active_nodes(wg)
    return {n: 0 for n in active_nodes}


def _leiden_igraph(wg: WindowGraph, cfg: CommunityConfig, ig) -> Dict[int, int]:
    """
    igraph Leiden on the directed weighted adjacency.

    igraph Leiden supports directed=True — no symmetrization needed for the
    primary path.  community_multilevel is undirected; it is used only if
    Leiden raises an error (e.g. disconnected graph edge case).
    """
    A = wg.A
    rows, cols = A.nonzero()
    weights = np.asarray(A[rows, cols]).flatten().astype(float).tolist()

    # igraph edge list
    edges = list(zip(rows.tolist(), cols.tolist()))
    G = ig.Graph(n=wg.n_nodes, edges=edges, directed=True)
    G.es["weight"] = weights

    try:
        # Directed Leiden (igraph >= 0.10)
        partition = G.community_leiden(
            weights="weight",
            resolution=cfg.resolution,
            directed=True,
        )
    except Exception:
        # Fallback: multilevel (undirected) — symmetrize and document
        # NOTE: direction is lost in this fallback. Weights are averaged.
        A_sym = symmetrize(A)
        rs, cs = A_sym.nonzero()
        ws = np.asarray(A_sym[rs, cs]).flatten().astype(float).tolist()
        G_u = ig.Graph(
            n=wg.n_nodes,
            edges=list(zip(rs.tolist(), cs.tolist())),
            directed=False,
        )
        G_u.es["weight"] = ws
        partition = G_u.community_multilevel(weights="weight")
        del G_u

    labels = {}
    for cid, members in enumerate(partition):
        for node in members:
            labels[node] = cid

    del G
    gc.collect()
    return labels


def _leiden_networkx(wg: WindowGraph, cfg: CommunityConfig, nx) -> Dict[int, int]:
    """
    networkx greedy_modularity fallback (undirected).
    Symmetrization applied — documented here.
    NOTE: direction is lost in this path. Use only when igraph is absent.
    """
    A_sym = symmetrize(wg.A)
    G = nx.from_scipy_sparse_array(A_sym, create_using=nx.Graph())
    communities = nx.community.greedy_modularity_communities(G, weight="weight")
    labels = {}
    for cid, members in enumerate(communities):
        for node in members:
            labels[int(node)] = cid
    del G, A_sym
    gc.collect()
    return labels


# ---------------------------------------------------------------------------
# Infomap path
# ---------------------------------------------------------------------------

def _run_infomap(wg: WindowGraph, cfg: CommunityConfig) -> Dict[int, int]:
    """
    Run Infomap for flow-based community detection (community_detection.md §6).
    Falls back to Leiden if infomap package is not installed.
    """
    im = _try_infomap()
    if im is None:
        return _run_leiden(wg, cfg)

    infomapper = im.Infomap("--directed --silent")
    A = wg.A
    rows, cols = A.nonzero()
    weights = np.asarray(A[rows, cols]).flatten()

    for u, v, w in zip(rows.tolist(), cols.tolist(), weights.tolist()):
        infomapper.add_link(u, v, w)

    infomapper.run()
    labels = {int(node.node_id): int(node.module_id) for node in infomapper.nodes}
    return labels


# ---------------------------------------------------------------------------
# Post-processing helpers
# ---------------------------------------------------------------------------

def _active_nodes(wg: WindowGraph) -> List[int]:
    """Return list of node IDs that have at least one edge in the window."""
    rows, cols = wg.A.nonzero()
    return list(set(rows.tolist()) | set(cols.tolist()))


def _filter_small_communities(
    labels: Dict[int, int],
    min_size: int,
) -> Dict[int, int]:
    """
    Reassign nodes in communities smaller than min_size to community -1.
    Does not renumber the remaining community IDs.
    """
    from collections import Counter
    counts = Counter(labels.values())
    return {
        node: (cid if counts[cid] >= min_size else -1)
        for node, cid in labels.items()
    }


def labels_to_dataframe(
    labels: Dict[int, int],
    window_id: int,
) -> pd.DataFrame:
    """
    Convert a labels dict to a tidy DataFrame for downstream use.

    Parameters
    ----------
    labels : Dict[node_id, community_id]
    window_id : int

    Returns
    -------
    pd.DataFrame with columns: window_id, node_id, community_id
    """
    if not labels:
        return pd.DataFrame(columns=["window_id", "node_id", "community_id"])
    df = pd.DataFrame(
        [(window_id, node, cid) for node, cid in labels.items()],
        columns=["window_id", "node_id", "community_id"],
    )
    return df.sort_values(["community_id", "node_id"]).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Node role classification
# ---------------------------------------------------------------------------

def compute_node_roles(
    window_df: pd.DataFrame,
    cfg: CommunityConfig,
    src_col: str = "src_node",
    dst_col: str = "dst_node",
    amount_col: str = "amount",
    alert_col: str = "is_sar",
) -> pd.DataFrame:
    """
    Classify each node in the window as source / sink / layering / neutral.

    Role encoding:
        0 = Neutral      (balanced or low-activity)
        1 = Source       (net_flow_ratio > role_threshold)
        2 = Sink         (net_flow_ratio < -role_threshold)
        3 = Layering     (flow_consistency > layering_consistency and
                          |net_flow_ratio| < role_threshold × 0.67)

    Layering score = flow_consistency × (1 - |net_flow_ratio|)

    Parameters
    ----------
    window_df : pd.DataFrame
        Single window of transactions. Must have src_col, dst_col, amount_col.
        alert_col is optional.
    cfg : CommunityConfig
    src_col, dst_col, amount_col, alert_col : str

    Returns
    -------
    pd.DataFrame with columns:
        node, total_volume, total_volume_out, total_volume_in,
        n_tx, n_alert_tx, alert_rate, net_flow_ratio,
        flow_consistency, layering_score, role
    """
    """
    Classify each node in the window as source / sink / layering / neutral.

    Role encoding:
        0 = Neutral      (balanced or low-activity)
        1 = Source       (net_flow_ratio > role_threshold)
        2 = Sink         (net_flow_ratio < -role_threshold)
        3 = Layering     (flow_consistency > layering_consistency and
                          |net_flow_ratio| < role_threshold × 0.67)

    Implementation Fix: Uses explicit priority assignment to avoid
    undefined role codes from overlapping thresholds.
    """
    has_alert = alert_col in window_df.columns

    # 1. Aggregation: Compute out-flow statistics per node
    agg_src = {"amt_out": (amount_col, "sum"), "n_out": (amount_col, "count")}
    if has_alert:
        agg_src["n_alert_out"] = (alert_col, "sum")
    src_stats = window_df.groupby(src_col, as_index=False).agg(**agg_src)
    src_stats = src_stats.rename(columns={src_col: "node"})

    # 2. Aggregation: Compute in-flow statistics per node
    agg_dst = {"amt_in": (amount_col, "sum"), "n_in": (amount_col, "count")}
    if has_alert:
        agg_dst["n_in_tx"] = (amount_col, "count") # Using count for tx volume
        if has_alert:
            agg_dst["n_alert_in"] = (alert_col, "sum")
    dst_stats = window_df.groupby(dst_col, as_index=False).agg(**agg_dst)
    dst_stats = dst_stats.rename(columns={dst_col: "node"})

    # 3. Merge: Align in-flow and out-flow on the same node ID
    nf = src_stats.merge(dst_stats, on="node", how="outer").fillna(0.0)

    # 4. Feature Engineering: Basic metrics
    nf["total_volume_out"] = nf["amt_out"].astype("float32")
    nf["total_volume_in"]  = nf["amt_in"].astype("float32")
    nf["total_volume"]     = nf["total_volume_out"] + nf["total_volume_in"]

    # Standardize transaction counts
    n_out_vals = nf["n_out"].astype("int32")
    n_in_vals = nf.get("n_in", nf.get("n_in_tx", 0)).astype("int32")
    nf["n_tx"] = n_out_vals + n_in_vals

    # Alert rate calculation
    if has_alert:
        nf["n_alert_tx"] = (nf.get("n_alert_out", 0) + nf.get("n_alert_in", 0)).astype("int32")
    else:
        nf["n_alert_tx"] = 0

    nf["alert_rate"] = nf["n_alert_tx"] / (nf["n_tx"] + 1e-9).astype("float32")

    # 5. Feature Engineering: Advanced AML metrics
    # Net flow ratio: balance between money coming in vs going out
    nf["net_flow_ratio"] = (
        (nf["total_volume_out"] - nf["total_volume_in"])
        / (nf["total_volume_out"] + nf["total_volume_in"] + 1e-9)
    ).astype("float32")

    # Flow consistency: high when in-flow and out-flow are nearly equal
    vol_min = nf[["total_volume_in", "total_volume_out"]].min(axis=1)
    nf["flow_consistency"] = (
        2 * vol_min / (nf["total_volume_in"] + nf["total_volume_out"] + 1e-9)
    ).astype("float32")

    # Layering score: derived from consistency and capital preservation
    nf["layering_score"] = (
        nf["flow_consistency"] * (1 - nf["net_flow_ratio"].abs())
    ).astype("float32")

    # 6. Role Assignment: Explicit Priority Logic (The Fix)
    # Define boolean masks based on Config thresholds
    is_source   = nf["net_flow_ratio"] >  cfg.role_threshold
    is_sink     = nf["net_flow_ratio"] < -cfg.role_threshold
    is_layering = (
        (nf["flow_consistency"] >  cfg.layering_consistency)
        & (nf["net_flow_ratio"].abs() < cfg.role_threshold * 0.67)
    )

    # Initialize all as Neutral (0)
    nf["role"] = 0

    # Apply labels with clear hierarchy.
    # Last assignment wins if a node satisfies multiple conditions.
    nf.loc[is_source,   "role"] = 1
    nf.loc[is_sink,     "role"] = 2
    nf.loc[is_layering, "role"] = 3  # Layering has highest priority in AML context

    return nf[[
        "node", "total_volume", "total_volume_out", "total_volume_in",
        "n_tx", "n_alert_tx", "alert_rate", "net_flow_ratio",
        "flow_consistency", "layering_score", "role",
    ]].reset_index(drop=True)



# ---------------------------------------------------------------------------
# Relay edge table (for edge-level analysis, optional)
# ---------------------------------------------------------------------------

def build_relay_edges(
    snapshot_edges: pd.DataFrame,
    src_col: str = "src_node",
    dst_col: str = "dst_node",
    weight_col: str = "weight",
    tx_count_col: str = "tx_count",
) -> pd.DataFrame:
    """
    Produce a directed relay edge table from snapshot edges.

    Keeps direction. Aggregates weight and transaction count per (src, dst) pair.
    Used for edge-level feature engineering and visualisation.

    Parameters
    ----------
    snapshot_edges : pd.DataFrame
        Output of build_snapshot_edges() from src/graph/temporal.py.
    src_col, dst_col, weight_col, tx_count_col : str

    Returns
    -------
    pd.DataFrame with columns:
        src, dst, weight, tx_count, alert_ratio (0.0 if no alert column)
    """
    if len(snapshot_edges) == 0:
        return pd.DataFrame(columns=["src", "dst", "weight", "tx_count"])

    agg = snapshot_edges.rename(columns={src_col: "src", dst_col: "dst"})
    keep = ["src", "dst", weight_col]
    if tx_count_col in agg.columns:
        keep.append(tx_count_col)
    agg = agg[keep]

    grouped = agg.groupby(["src", "dst"], as_index=False).agg(
        weight=(weight_col, "sum"),
        tx_count=(tx_count_col if tx_count_col in agg.columns else weight_col, "count"),
    )

    return grouped.reset_index(drop=True)


# ---------------------------------------------------------------------------
# Recursive community splitting (oversized community guard)
# ---------------------------------------------------------------------------

def split_large_communities(
    labels: Dict[int, int],
    wg: WindowGraph,
    cfg: CommunityConfig,
    depth: int = 0,
) -> Dict[int, int]:
    """
    Recursively split communities larger than cfg.s_max.

    Extracts the subgraph of each oversized community and runs
    detect_communities on it independently.  Assigns globally unique
    community IDs across all splits.

    Parameters
    ----------
    labels : Dict[node_id, community_id]
        Initial community assignment.
    wg : WindowGraph
        The full window graph (used to extract subgraph weights).
    cfg : CommunityConfig
    depth : int
        Current recursion depth (stops at cfg.max_recursion_depth).

    Returns
    -------
    Dict[node_id, community_id] — refined labels with splits applied.
    """
    if depth >= cfg.max_recursion_depth:
        return labels

    from collections import defaultdict
    comm_nodes: Dict[int, List[int]] = defaultdict(list)
    for node, cid in labels.items():
        comm_nodes[cid].append(node)

    oversized = {cid: nodes for cid, nodes in comm_nodes.items()
                 if len(nodes) > cfg.s_max}

    if not oversized:
        return labels

    max_cid = max(labels.values()) + 1
    result = dict(labels)

    # Pre-compute full nonzero coordinates once outside the per-community loop
    all_rows, all_cols = wg.A.nonzero()

    for cid, members in oversized.items():
        node_set = set(members)
        # Extract subgraph CSR for this community
        mask = np.isin(all_rows, list(node_set)) & np.isin(all_cols, list(node_set))
        sub_rows = all_rows[mask]
        sub_cols = all_cols[mask]

        if len(sub_rows) == 0:
            continue

        # Read through matrix interface — safe regardless of CSR internal layout
        sub_data = np.asarray(wg.A[sub_rows, sub_cols]).flatten().astype(np.float32)

        # Remap to local indices
        sorted_members = sorted(members)
        idx_map = {n: i for i, n in enumerate(sorted_members)}
        local_rows = np.array([idx_map[r] for r in sub_rows])
        local_cols = np.array([idx_map[c] for c in sub_cols])
        from scipy.sparse import csr_matrix as _csr
        sub_A = _csr((sub_data, (local_rows, local_cols)),
                     shape=(len(sorted_members), len(sorted_members)),
                     dtype=np.float32)

        from .weighting import WindowGraph as WG
        sub_wg = WG(
            A=sub_A,
            d_out=np.asarray(sub_A.sum(axis=1)).flatten(),
            d_in=np.asarray(sub_A.sum(axis=0)).flatten(),
            m_t=float(sub_data.sum()),
            n_nodes=len(sorted_members),
        )

        sub_labels_local = detect_communities(sub_wg, cfg)
        sub_labels_local = split_large_communities(sub_labels_local, sub_wg, cfg, depth + 1)

        # Remap back to global node IDs
        rev_map = {i: n for n, i in idx_map.items()}
        for local_node, local_cid in sub_labels_local.items():
            global_node = rev_map[local_node]
            result[global_node] = max_cid + local_cid

        max_cid += (max(sub_labels_local.values()) + 1 if sub_labels_local else 1)

    return result


__all__ = [
    "detect_communities",
    "compute_node_roles",
    "build_relay_edges",
    "labels_to_dataframe",
    "split_large_communities",
    # Legacy names kept for __init__.py compatibility
    "run_recursive_leiden",
]


In [ ]:
"""
 AML-aware community suspicion scoring.

Two public functions:

    extract_community_features(window_df, snap_edges, global_labels, wg, node_roles)
        Compute AML-relevant features for each community in one window.
        Called once per window by pipeline.py — never accumulates all windows.

    score_communities(feature_df, cfg, persistence_map)
        Apply the weighted suspicion formula to the feature table.
        Returns the same DataFrame with added score columns.

Score formula (community_scoring.md §2):
    S(C) = w_internal_flow  * InternalFlow(C)
         + w_reciprocity    * Reciprocity(C)
         + w_persistence    * Persistence(C)
         + w_motif          * MotifEnrichment(C)
         - w_external_noise * ExternalNoise(C)

Feature definitions:
    internal_flow    : total_internal_weight / total_weight (in window)
    reciprocity      : bidirectional_volume / total_internal_volume
    persistence      : n_windows_seen / max_possible_windows (0–1, from tracker)
    motif_enrichment : motif_count_inside / community_size (0 if no motif table)
    external_noise   : total_external_weight / total_weight (1 - internal_flow)

Guarantees:
    - Time      : per-window; no cross-window state inside this module.
    - Direction : A[i,j] used directly for internal/external flow; no symmetrization.
    - Memory    : all operations are vectorized pandas groupby; no cuDF/cupy.
"""

from __future__ import annotations

import gc
from typing import Dict, List, Optional

import numpy as np
import pandas as pd




# ---------------------------------------------------------------------------
# Per-window feature extraction
# ---------------------------------------------------------------------------

def extract_community_features(
    window_df: pd.DataFrame,
    global_labels: Dict[int, int],
    wg: WindowGraph,
    node_roles: Optional[pd.DataFrame] = None,
    motif_counts: Optional[Dict[int, float]] = None,
    window_id: int = 0,
    step_start: int = 0,
    step_end: int = 0,
    src_col: str = "src_node",
    dst_col: str = "dst_node",
    amount_col: str = "amount",
    alert_col: str = "is_sar",
) -> pd.DataFrame:
    """
    Extract AML community features for one time window.

    Parameters
    ----------
    window_df : pd.DataFrame
        Raw transaction rows for this window. Provides amount and alert info.
    global_labels : Dict[node_id, global_cid]
        Output of match_communities_jaccard() for this window.
    wg : WindowGraph
        Directed weighted adjacency for this window (from build_window_graph).
    node_roles : pd.DataFrame, optional
        Output of compute_node_roles(). Must have columns: node, role,
        flow_consistency, layering_score.
    motif_counts : Dict[global_cid, float], optional
        Motif instance count per community (from motif module integration).
        If None, motif_enrichment is 0 for all communities.
    window_id, step_start, step_end : int
        Window metadata stored in the output.
    src_col, dst_col, amount_col, alert_col : str
        Column names in window_df.

    Returns
    -------
    pd.DataFrame with one row per active community. Columns:
        window_id, step_start, step_end, global_cid,
        size, internal_flow, external_flow, external_noise,
        reciprocity, internal_recirc,
        sink_concentration, source_concentration,
        n_sources, n_sinks, n_layering, avg_layering_score,
        total_volume, n_internal_edges,
        alert_ratio, motif_enrichment, vol_density, edge_density,
        [persistence is added later by score_communities]
    """
    # Filter out noise nodes (-1 community)
    active = {n: c for n, c in global_labels.items() if c != -1}
    if not active:
        return pd.DataFrame()

    node_to_cid = pd.Series(active, name="global_cid")
    node_to_cid.index.name = "node"
    node_to_cid = node_to_cid.reset_index()

    # ── Map edges to communities ─────────────────────────────────────────────
    has_alert = alert_col in window_df.columns
    agg_dict = {
        "weight": (amount_col, "sum"),
        "n_tx": (amount_col, "count"),
    }
    if has_alert:
        agg_dict["n_alert"] = (alert_col, "sum")

    edges = (
        window_df.groupby([src_col, dst_col], as_index=False)
        .agg(**agg_dict)
        .rename(columns={src_col: "src", dst_col: "dst"})
    )

    edges = edges.merge(
        node_to_cid.rename(columns={"node": "src", "global_cid": "src_cid"}),
        on="src", how="inner",
    )
    edges = edges.merge(
        node_to_cid.rename(columns={"node": "dst", "global_cid": "dst_cid"}),
        on="dst", how="inner",
    )

    total_weight = float(edges["weight"].sum()) if len(edges) > 0 else 1.0

    internal = edges[edges["src_cid"] == edges["dst_cid"]].copy()
    internal = internal.rename(columns={"src_cid": "global_cid"}).drop(columns=["dst_cid"])

    external = edges[edges["src_cid"] != edges["dst_cid"]].copy()

    # ── Basic internal features ───────────────────────────────────────────────
    if len(internal) == 0:
        return pd.DataFrame()

    if has_alert:
        internal["is_alert_edge"] = (internal["n_alert"] > 0).astype("float32")
        comm_basic = internal.groupby("global_cid", as_index=False).agg(
            total_volume      =("weight",        "sum"),
            n_internal_edges  =("weight",        "count"),
            alert_ratio       =("is_alert_edge", "mean"),
        )
    else:
        comm_basic = internal.groupby("global_cid", as_index=False).agg(
            total_volume      =("weight",   "sum"),
            n_internal_edges  =("weight",   "count"),
        )
        comm_basic["alert_ratio"] = 0.0

    # ── External flow ─────────────────────────────────────────────────────────
    ext_agg = external.groupby("src_cid", as_index=False).agg(
        ext_volume=("weight", "sum"),
    ).rename(columns={"src_cid": "global_cid"})
    comm_basic = comm_basic.merge(ext_agg, on="global_cid", how="left")
    comm_basic["ext_volume"] = comm_basic["ext_volume"].fillna(0.0)
    comm_basic["internal_flow"]   = (comm_basic["total_volume"] / (total_weight + 1e-9)).astype("float32")
    comm_basic["external_flow"]   = (comm_basic["ext_volume"]   / (total_weight + 1e-9)).astype("float32")
    comm_basic["external_noise"]  = comm_basic["external_flow"]

    # ── Reciprocity (bidirectional volume ratio) ───────────────────────────────
    reverse = internal[["src", "dst", "global_cid", "weight"]].rename(
        columns={"src": "dst_r", "dst": "src_r", "weight": "rev_w"}
    )
    recirc = internal[["src", "dst", "global_cid", "weight"]].merge(
        reverse,
        left_on=["src", "dst", "global_cid"],
        right_on=["src_r", "dst_r", "global_cid"],
        how="inner",
    )
    if len(recirc) > 0:
        recirc_agg = recirc.groupby("global_cid", as_index=False).agg(
            recirc_volume=("weight", "sum")
        )
    else:
        recirc_agg = pd.DataFrame({"global_cid": pd.Series(dtype="int64"),
                                   "recirc_volume": pd.Series(dtype="float32")})

    comm_basic = comm_basic.merge(recirc_agg, on="global_cid", how="left")
    comm_basic["recirc_volume"] = comm_basic["recirc_volume"].fillna(0.0)
    comm_basic["reciprocity"] = (
        comm_basic["recirc_volume"] / (comm_basic["total_volume"] + 1e-9)
    ).astype("float32")
    comm_basic["internal_recirc"] = comm_basic["reciprocity"]
    comm_basic = comm_basic.drop(columns=["recirc_volume", "ext_volume"])

    # ── Sink / source concentration ───────────────────────────────────────────
    node_in = internal.groupby(["global_cid", "dst"], as_index=False).agg(in_vol=("weight", "sum"))
    max_in  = node_in.groupby("global_cid", as_index=False).agg(max_in=("in_vol", "max"))
    sum_in  = node_in.groupby("global_cid", as_index=False).agg(sum_in=("in_vol", "sum"))
    sink_c  = max_in.merge(sum_in, on="global_cid")
    sink_c["sink_concentration"] = (sink_c["max_in"] / (sink_c["sum_in"] + 1e-9)).astype("float32")
    comm_basic = comm_basic.merge(sink_c[["global_cid", "sink_concentration"]], on="global_cid", how="left")

    node_out  = internal.groupby(["global_cid", "src"], as_index=False).agg(out_vol=("weight", "sum"))
    max_out   = node_out.groupby("global_cid", as_index=False).agg(max_out=("out_vol", "max"))
    sum_out   = node_out.groupby("global_cid", as_index=False).agg(sum_out=("out_vol", "sum"))
    source_c  = max_out.merge(sum_out, on="global_cid")
    source_c["source_concentration"] = (source_c["max_out"] / (source_c["sum_out"] + 1e-9)).astype("float32")
    comm_basic = comm_basic.merge(source_c[["global_cid", "source_concentration"]], on="global_cid", how="left")

    # ── Community size ────────────────────────────────────────────────────────
    sizes = node_to_cid.groupby("global_cid", as_index=False).agg(size=("node", "count"))
    comm_basic = comm_basic.merge(sizes, on="global_cid", how="left")

    # ── Node roles ────────────────────────────────────────────────────────────
    if node_roles is not None and len(node_roles) > 0:
        nr = node_roles[["node", "role", "flow_consistency", "layering_score"]].merge(
            node_to_cid, on="node", how="inner"
        )
        nr["is_source"]   = (nr["role"] == 1).astype("int32")
        nr["is_sink"]     = (nr["role"] == 2).astype("int32")
        nr["is_layering"] = (nr["role"] == 3).astype("int32")
        role_agg = nr.groupby("global_cid", as_index=False).agg(
            n_sources          =("is_source",       "sum"),
            n_sinks            =("is_sink",          "sum"),
            n_layering         =("is_layering",      "sum"),
            avg_layering_score =("layering_score",   "mean"),
        )
        comm_basic = comm_basic.merge(role_agg, on="global_cid", how="left")
    else:
        for col in ["n_sources", "n_sinks", "n_layering", "avg_layering_score"]:
            comm_basic[col] = 0

    # ── Motif enrichment ──────────────────────────────────────────────────────
    if motif_counts:
        # Pre-build size lookup to avoid O(n²) per-row DataFrame filter
        # and IndexError when a community is in motif_counts but not comm_basic.
        size_dict = comm_basic.set_index("global_cid")["size"].to_dict()
        comm_basic["motif_enrichment"] = comm_basic["global_cid"].map(
            lambda gcid: motif_counts.get(gcid, 0.0) / max(size_dict.get(gcid, 1), 1)
        ).astype("float32")
    else:
        comm_basic["motif_enrichment"] = 0.0

    # ── Size-normalised features ───────────────────────────────────────────────
    comm_basic["vol_density"]  = (comm_basic["total_volume"] / (comm_basic["size"] + 1e-9)).astype("float32")
    comm_basic["edge_density"] = (
        comm_basic["n_internal_edges"].astype("float32")
        / (comm_basic["size"] * (comm_basic["size"] - 1) + 1e-9)
    ).astype("float32")

    # ── Metadata ──────────────────────────────────────────────────────────────
    comm_basic["window_id"]  = window_id
    comm_basic["step_start"] = step_start
    comm_basic["step_end"]   = step_end
    comm_basic["persistence"] = 0.0   # filled by score_communities from persistence_map

    # Fill remaining NaN
    for col in comm_basic.select_dtypes(include=[np.number]).columns:
        comm_basic[col] = comm_basic[col].fillna(0.0)

    # Filter by min community size
    return comm_basic[comm_basic["size"] >= 1].reset_index(drop=True)


# ---------------------------------------------------------------------------
# Suspicion scoring
# ---------------------------------------------------------------------------

def score_communities(
    feature_df: pd.DataFrame,
    cfg: CommunityConfig,
    persistence_map: Optional[Dict[int, int]] = None,
    max_windows: int = 1,
    n_alert: int = 0,
    n_rows: int = 1,
) -> pd.DataFrame:
    """
    Compute suspicion scores for all communities in feature_df.

    Formula (community_scoring.md §2):
        S(C) = w_internal_flow  * internal_flow
             + w_reciprocity    * reciprocity
             + w_persistence    * persistence_norm
             + w_motif          * motif_enrichment_norm
             - w_external_noise * external_noise

    Then apply a log size-penalty to avoid large communities dominating.

    Parameters
    ----------
    feature_df : pd.DataFrame
        Output of extract_community_features() — one or many windows concatenated.
    cfg : CommunityConfig
        Score weights and threshold.
    persistence_map : Dict[global_cid, int], optional
        {global_cid: n_windows_seen}. If None, persistence = 0 for all.
    max_windows : int
        Total number of windows processed — used to normalise persistence to [0, 1].
    n_alert : int
        Total SAR-labelled transactions in the dataset (for AER reporting).
    n_rows : int
        Total transactions in the dataset (for AER reporting).

    Returns
    -------
    pd.DataFrame — same as input, with added columns:
        persistence, persistence_norm, motif_enrichment_norm,
        suspicion_score, is_suspicious,
        flag_internal_flow, flag_reciprocity, flag_persistence, flag_motif
    """
    if len(feature_df) == 0:
        return feature_df

    df = feature_df.copy()

    # ── Persistence normalisation ─────────────────────────────────────────────
    if persistence_map:
        df["persistence"] = df["global_cid"].map(
            lambda gcid: persistence_map.get(gcid, 0)
        ).astype("float32")
    # persistence_norm ∈ [0, 1]
    df["persistence_norm"] = (df["persistence"] / max(max_windows, 1)).clip(0.0, 1.0).astype("float32")

    # ── Motif enrichment normalisation ────────────────────────────────────────
    max_motif = float(df["motif_enrichment"].max()) if df["motif_enrichment"].max() > 0 else 1.0
    df["motif_enrichment_norm"] = (df["motif_enrichment"] / max_motif).clip(0.0, 1.0).astype("float32")

    # ── Weighted suspicion score ───────────────────────────────────────────────
    df["suspicion_score"] = (
        cfg.w_internal_flow  * df["internal_flow"].clip(0.0, 1.0)
        + cfg.w_reciprocity  * df["reciprocity"].clip(0.0, 1.0)
        + cfg.w_persistence  * df["persistence_norm"]
        + cfg.w_motif        * df["motif_enrichment_norm"]
        - cfg.w_external_noise * df["external_noise"].clip(0.0, 1.0)
    ).astype("float32")

    # Size-penalty: log penalty to avoid large communities dominating.
    # gamma is configurable via cfg.size_penalty_gamma (default 0.05).
    gamma = cfg.size_penalty_gamma
    size_penalty = gamma * np.log1p(df["size"].astype("float64")).astype("float32")
    df["suspicion_score"] = (df["suspicion_score"] - size_penalty).clip(lower=0.0).astype("float32")

    # ── Binary flag ───────────────────────────────────────────────────────────
    df["is_suspicious"] = (df["suspicion_score"] >= cfg.global_susp_threshold).astype("int8")

    # ── Explainability flags ───────────────────────────────────────────────────
    df["flag_internal_flow"] = (df["internal_flow"]  >= 0.5).astype("int8")
    df["flag_reciprocity"]   = (df["reciprocity"]    >= 0.3).astype("int8")
    df["flag_persistence"]   = (df["persistence_norm"] >= 0.3).astype("int8")
    df["flag_motif"]         = (df["motif_enrichment"] >= 1.0).astype("int8")

    # ── Summary ────────────────────────────────────────────────────────────────
    total  = len(df)
    n_susp = int(df["is_suspicious"].sum())
    global_sar_rate = n_alert / max(n_rows, 1)
    susp_sar_rate   = float(df[df["is_suspicious"] == 1]["alert_ratio"].mean()) if n_susp > 0 else 0.0
    aer = susp_sar_rate / max(global_sar_rate, 1e-9)

    print(f"  Communities: {total:,} | Suspicious: {n_susp:,} ({n_susp / max(total, 1) * 100:.1f}%)")
    print(f"  Mean score: {float(df['suspicion_score'].mean()):.4f}")
    if n_alert > 0:
        print(f"  AER: {aer:.2f}×  (global={global_sar_rate * 100:.2f}%, susp={susp_sar_rate * 100:.2f}%)")

    unknown_unknowns = df[
        (df["is_suspicious"] == 1) & (df["alert_ratio"] < cfg.alert_thresh)
    ]
    if len(unknown_unknowns) > 0:
        print(f"  Unknown-unknowns (suspicious, alert_ratio < {cfg.alert_thresh}): {len(unknown_unknowns)}")

    return df.sort_values("suspicion_score", ascending=False).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Shortlist helper
# ---------------------------------------------------------------------------

def get_shortlist(
    scored_df: pd.DataFrame,
    cfg: CommunityConfig,
) -> pd.DataFrame:
    """
    Return the top-K suspicious communities for investigation.

    Parameters
    ----------
    scored_df : pd.DataFrame
        Output of score_communities().
    cfg : CommunityConfig
        top_k_export and global_susp_threshold.

    Returns
    -------
    pd.DataFrame — top-K rows from scored_df, filtered to is_suspicious == 1.
    """
    suspicious = scored_df[scored_df["is_suspicious"] == 1]
    return suspicious.head(cfg.top_k_export).reset_index(drop=True)


__all__ = [
    "extract_community_features",
    "score_communities",
    "get_shortlist",
]

In [ ]:
"""
 Cross-window community identity tracking.

Assigns persistent community IDs across consecutive time windows using
Jaccard-based node overlap matching.

Public API:

    match_communities_jaccard(curr_labels, tracking_buffer, cfg, counter)
        Core matching function. Compares current window's communities
        against a rolling buffer of recent windows.
        Returns (new_labels_with_global_ids, updated_counter).

    build_tracking_record(global_labels, window_id, step_start, step_end)
        Produces the per-window tracking output table (tracking.md §4).

    update_buffer(buffer, record_df, cfg)
        Maintain the rolling tracking buffer (capped at tracking_memory).

Matching rules (tracking.md §2–5):
    - Jaccard(C_a, C_b) = |C_a ∩ C_b| / |C_a ∪ C_b|
    - If Jaccard >= cfg.jaccard_thresh  → inherit persistent ID from best match
    - Otherwise                         → assign new persistent ID
    - Buffer spans cfg.tracking_memory past windows (oldest first).
      A community that "disappears" for ≤ tracking_memory windows can still
      match when it re-appears.

Split / merge handling (tracking.md §5):
    - Dominant overlap path: a current community inherits the ID of
      whichever past community it overlaps with most strongly.
    - If two current communities both best-match the same past community,
      the one with higher Jaccard takes the ID; the other gets a new ID.

Guarantees:
    - Time      : processing is sequential; buffer contains only past windows.
    - Direction : node IDs come from detection.py which uses WindowGraph (directed).
    - Memory    : buffer is capped; only node→global_cid mappings retained (not full graphs).
"""

from __future__ import annotations

import gc
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd




# ---------------------------------------------------------------------------
# Core matcher
# ---------------------------------------------------------------------------

def match_communities_jaccard(
    curr_labels: Dict[int, int],
    tracking_buffer: List[pd.DataFrame],
    cfg: CommunityConfig,
    global_cid_counter: int,
) -> Tuple[Dict[int, int], int]:
    """
    Assign persistent (global) community IDs to the current window's communities.

    Parameters
    ----------
    curr_labels : Dict[node_id, local_community_id]
        Output of detect_communities() for the current window.
        Local community IDs are window-local (not persistent).
    tracking_buffer : List[pd.DataFrame]
        Each entry is a DataFrame with columns [node, global_cid] from a
        past window. Oldest first; length capped at cfg.tracking_memory.
    cfg : CommunityConfig
        jaccard_thresh and tracking_memory are used here.
    global_cid_counter : int
        Next unused persistent community ID.

    Returns
    -------
    (global_labels, updated_counter)
        global_labels : Dict[node_id, global_community_id]
        updated_counter : int — new value after assigning any new IDs.

    Notes
    -----
    - Only plain Python sets and pandas operations; no cuDF.
    - When two current communities best-match the same past community,
      the higher-Jaccard one inherits the ID; the other gets a new one.
    - Nodes filtered out by detect_communities (cid == -1) keep cid == -1.
    """
    # Separate out noise / filtered nodes (community -1)
    active = {n: c for n, c in curr_labels.items() if c != -1}

    if not active:
        return curr_labels, global_cid_counter

    # Group current nodes by local community ID
    curr_sets: Dict[int, set] = {}
    for node, cid in active.items():
        curr_sets.setdefault(cid, set()).add(node)

    local_cids = list(curr_sets.keys())

    # No history → assign fresh IDs to all
    if not tracking_buffer:
        mapping = {}
        for cid in local_cids:
            mapping[cid] = global_cid_counter
            global_cid_counter += 1
        global_labels = {n: (mapping[c] if c != -1 else -1) for n, c in curr_labels.items()}
        return global_labels, global_cid_counter

    # Build node → global_cid lookup from buffer (most-recent window wins)
    prev_lookup: Dict[int, int] = {}
    for buf_df in tracking_buffer:           # oldest first
        for _, row in buf_df.iterrows():
            prev_lookup[int(row["node"])] = int(row["global_cid"])

    # Group previous nodes by global_cid
    prev_sets: Dict[int, set] = {}
    for node, gcid in prev_lookup.items():
        prev_sets.setdefault(gcid, set()).add(node)

    prev_gcids = list(prev_sets.keys())

    # Compute Jaccard for all (curr_cid, prev_gcid) pairs — vectorized
    best_match: Dict[int, Tuple[int, float]] = {}   # local_cid → (best_gcid, best_j)

    for local_cid, curr_nodes in curr_sets.items():
        best_j    = -1.0
        best_gcid = None

        for gcid, prev_nodes in prev_sets.items():
            inter = len(curr_nodes & prev_nodes)
            if inter == 0:
                continue
            union = len(curr_nodes | prev_nodes)
            j = inter / union
            if j > best_j:
                best_j    = j
                best_gcid = gcid

        if best_gcid is not None and best_j >= cfg.jaccard_thresh:
            best_match[local_cid] = (best_gcid, best_j)

    # Resolve conflicts: two local communities claim the same past global ID
    claimed: Dict[int, Tuple[int, float]] = {}     # gcid → (local_cid, jaccard)
    for local_cid, (gcid, j) in best_match.items():
        if gcid not in claimed or j > claimed[gcid][1]:
            claimed[gcid] = (local_cid, j)

    # Build local_cid → global_cid mapping
    winner_locals = {local_cid for (local_cid, _) in claimed.values()}
    local_to_global: Dict[int, int] = {}

    for local_cid in local_cids:
        if local_cid in winner_locals:
            # Find which gcid this local_cid won
            for gcid, (winning_local, _) in claimed.items():
                if winning_local == local_cid:
                    local_to_global[local_cid] = gcid
                    break
        else:
            # No match or lost conflict → new persistent ID
            local_to_global[local_cid] = global_cid_counter
            global_cid_counter += 1

    global_labels = {
        n: (local_to_global[c] if c != -1 else -1)
        for n, c in curr_labels.items()
    }
    return global_labels, global_cid_counter


# ---------------------------------------------------------------------------
# Tracking record builder
# ---------------------------------------------------------------------------

def build_tracking_record(
    global_labels: Dict[int, int],
    window_id: int,
    step_start: int = 0,
    step_end: int = 0,
    prev_buffer: List[pd.DataFrame] | None = None,
) -> pd.DataFrame:
    """
    Build the per-window tracking output table (tracking.md §4).

    Output columns:
        window_id, step_start, step_end,
        node, global_cid, community_size, overlap_score

    overlap_score is the max Jaccard between this community and any community
    in the previous buffer entry.  0.0 if no buffer provided or no match.

    Parameters
    ----------
    global_labels : Dict[node_id, global_community_id]
    window_id : int
    step_start, step_end : int
    prev_buffer : List[pd.DataFrame], optional
        Used only to compute overlap_score for reporting.

    Returns
    -------
    pd.DataFrame with columns:
        window_id, step_start, step_end, node, global_cid,
        community_size, overlap_score
    """
    if not global_labels:
        return pd.DataFrame(columns=[
            "window_id", "step_start", "step_end", "node",
            "global_cid", "community_size", "overlap_score",
        ])

    # Community sizes
    size_map: Dict[int, int] = {}
    for gcid in global_labels.values():
        if gcid != -1:
            size_map[gcid] = size_map.get(gcid, 0) + 1

    # Overlap scores (optional, reported per community)
    overlap_map: Dict[int, float] = {}
    if prev_buffer:
        last = prev_buffer[-1]  # most recent past window
        # dict(zip(...)) is O(n) — avoids iterrows() overhead on large windows
        prev_lookup: Dict[int, int] = dict(zip(
            last["node"].astype(int), last["global_cid"].astype(int)
        ))

        prev_sets: Dict[int, set] = {}
        for node, gcid in prev_lookup.items():
            prev_sets.setdefault(gcid, set()).add(node)

        curr_sets: Dict[int, set] = {}
        for node, gcid in global_labels.items():
            if gcid != -1:
                curr_sets.setdefault(gcid, set()).add(node)

        for gcid, curr_nodes in curr_sets.items():
            best_j = 0.0
            for prev_nodes in prev_sets.values():
                inter = len(curr_nodes & prev_nodes)
                if inter == 0:
                    continue
                j = inter / len(curr_nodes | prev_nodes)
                if j > best_j:
                    best_j = j
            overlap_map[gcid] = round(best_j, 4)

    rows = []
    for node, gcid in global_labels.items():
        rows.append({
            "window_id":      window_id,
            "step_start":     step_start,
            "step_end":       step_end,
            "node":           int(node),
            "global_cid":     int(gcid),
            "community_size": size_map.get(gcid, 0),
            "overlap_score":  overlap_map.get(gcid, 0.0),
        })

    return pd.DataFrame(rows).sort_values(["global_cid", "node"]).reset_index(drop=True)


# ---------------------------------------------------------------------------
# Buffer management
# ---------------------------------------------------------------------------

def update_buffer(
    buffer: List[pd.DataFrame],
    record_df: pd.DataFrame,
    cfg: CommunityConfig,
) -> List[pd.DataFrame]:
    """
    Append the current window's node→global_cid mapping to the buffer,
    and trim to the last cfg.tracking_memory entries.

    Parameters
    ----------
    buffer : List[pd.DataFrame]
        Existing buffer (modified in-place conceptually, returned explicitly).
    record_df : pd.DataFrame
        Output of build_tracking_record() for the current window.
    cfg : CommunityConfig

    Returns
    -------
    Updated buffer list.
    """
    # Keep only [node, global_cid] for the buffer — discard window metadata
    if "node" in record_df.columns and "global_cid" in record_df.columns:
        slim = record_df[["node", "global_cid"]].copy()
        slim = slim[slim["global_cid"] != -1]   # skip noise nodes
    else:
        slim = pd.DataFrame(columns=["node", "global_cid"])

    buffer.append(slim)

    # Cap buffer size
    if len(buffer) > cfg.tracking_memory:
        removed = buffer.pop(0)
        del removed

    gc.collect()
    return buffer


__all__ = [
    "match_communities_jaccard",
    "build_tracking_record",
    "update_buffer",
]

In [ ]:
"""
 Build directed weighted graph representations for community detection.

Responsibility: take snapshot_edges for one time window → produce a WindowGraph
containing the sparse adjacency matrix and derived degree statistics.

Two public functions:

    build_window_graph(snapshot_edges, n_nodes, cfg)
        → WindowGraph  (primary path)

    symmetrize(A)
        → scipy.sparse.csr_matrix  (fallback when an algorithm requires undirected)

WindowGraph contains:
    A       : csr_matrix, shape (n_nodes, n_nodes), A[i,j] = total amount i→j
    d_out   : ndarray (n_nodes,) weighted out-degree
    d_in    : ndarray (n_nodes,) weighted in-degree
    m_t     : float, total edge weight in window
    n_nodes : int

Optional secondary function:

    apply_weighting(snapshot_edges, cfg)
        Applies monetary continuity and temporal decay factors to edge weights
        before building the graph.  Used when the config requests weighted edges
        beyond raw amounts.

Guarantees:
    - Direction : A[i,j] ≠ A[j,i] in general; d_out ≠ d_in in general.
                  symmetrize() is only called explicitly — never by default.
    - Time      : operates on a single window's edges; no cross-window state.
    - Memory    : CSR sparse (O(edges)), numpy 1-D arrays for degrees.
                  No dense adjacency matrix created.
"""

from __future__ import annotations

import gc
from dataclasses import dataclass

import numpy as np
from scipy.sparse import csr_matrix

try:
    import pandas as pd
    _GPU = False
    _pd = pd
except ImportError:  # should never happen — pandas is always available
    raise

try:
    import cudf as _cudf_mod
    _GPU = True
except ImportError:
    _cudf_mod = None


# ---------------------------------------------------------------------------
# Output container
# ---------------------------------------------------------------------------

@dataclass
class WindowGraph:
    """
    Directed weighted graph for one time window.

    Attributes
    ----------
    A       : csr_matrix of shape (n_nodes, n_nodes).
              A[i, j] = total amount transferred from i to j in this window.
              Not symmetrised.
    d_out   : ndarray (n_nodes,) — weighted out-degree per node.
    d_in    : ndarray (n_nodes,) — weighted in-degree per node.
    m_t     : float — total edge weight (sum of all amounts) in the window.
    n_nodes : int — matrix dimension.
    step_start : int — first step of the window.
    step_end   : int — last step of the window.
    """
    A:          csr_matrix
    d_out:      np.ndarray
    d_in:       np.ndarray
    m_t:        float
    n_nodes:    int
    step_start: int = 0
    step_end:   int = 0


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _to_numpy(series) -> np.ndarray:
    """Convert pandas or cuDF Series to numpy array."""
    if hasattr(series, "to_pandas"):
        return series.to_pandas().to_numpy()
    return series.to_numpy()


def _to_pandas(df) -> _pd.DataFrame:
    """Convert cuDF DataFrame to pandas if needed."""
    if hasattr(df, "to_pandas"):
        return df.to_pandas()
    return df


# ---------------------------------------------------------------------------
# Primary builder
# ---------------------------------------------------------------------------

def build_window_graph(
    snapshot_edges,
    n_nodes: int | None = None,
    cfg=None,
    step_start: int = 0,
    step_end: int = 0,
    src_col: str = "src_node",
    dst_col: str = "dst_node",
    weight_col: str = "weight",
) -> WindowGraph:
    """
    Build a directed weighted sparse graph from a window edge table.

    Per community_detection.md §2:
        A^(t)[i, j] = sum of all amounts from node i to node j in window t.

    Parameters
    ----------
    snapshot_edges : DataFrame (pandas or cuDF)
        Output of build_snapshot_edges() from src/graph/temporal.py.
        Must have src_col, dst_col, weight_col as integer encoded node IDs.
    n_nodes : int, optional
        Square matrix dimension.  Inferred as max(node_id) + 1 if None.
        Pass the encoder's n_nodes for consistent shape across windows.
    cfg : CommunityConfig, optional
        If provided, apply_weighting() is called on the edges first.
        If None, raw amounts are used directly.
    step_start, step_end : int
        Window boundaries stored in the WindowGraph for tracking.
    src_col, dst_col, weight_col : str
        Column names in snapshot_edges.

    Returns
    -------
    WindowGraph with A (CSR), d_out, d_in, m_t, n_nodes.

    Notes
    -----
    - Direction is preserved: A[i,j] ≠ A[j,i] in general.
    - No dense matrix is ever created.
    - If snapshot_edges is empty, returns a zero-filled WindowGraph.
    """
    edges = _to_pandas(snapshot_edges)

    if len(edges) == 0:
        dim = n_nodes or 0
        return WindowGraph(
            A=csr_matrix((dim, dim), dtype=np.float32),
            d_out=np.zeros(dim, dtype=np.float32),
            d_in=np.zeros(dim, dtype=np.float32),
            m_t=0.0,
            n_nodes=dim,
            step_start=step_start,
            step_end=step_end,
        )

    # Optional: apply monetary continuity + temporal decay weighting
    if cfg is not None:
        edges = apply_weighting(edges, cfg, src_col=src_col, dst_col=dst_col,
                                weight_col=weight_col)

    row  = edges[src_col].to_numpy().astype(np.int64)
    col  = edges[dst_col].to_numpy().astype(np.int64)
    data = edges[weight_col].to_numpy().astype(np.float32)

    if n_nodes is None:
        n_nodes = int(max(row.max(), col.max())) + 1

    A = csr_matrix((data, (row, col)), shape=(n_nodes, n_nodes), dtype=np.float32)

    # Degree vectors — CSR row-sum = out-degree; column-sum = in-degree
    d_out = np.asarray(A.sum(axis=1)).flatten()   # shape (n_nodes,)
    d_in  = np.asarray(A.sum(axis=0)).flatten()   # shape (n_nodes,)
    m_t   = float(data.sum())

    return WindowGraph(
        A=A,
        d_out=d_out,
        d_in=d_in,
        m_t=m_t,
        n_nodes=n_nodes,
        step_start=step_start,
        step_end=step_end,
    )


# ---------------------------------------------------------------------------
# Optional: apply monetary continuity + temporal decay to edge weights
# ---------------------------------------------------------------------------

def apply_weighting(
    edges: _pd.DataFrame,
    cfg,
    src_col: str = "src_node",
    dst_col: str = "dst_node",
    weight_col: str = "weight",
    avg_gap_col: str = "avg_gap",
    delta_w: int = 30,
) -> _pd.DataFrame:
    """
    Apply monetary continuity and temporal decay factors to snapshot edge weights.

    Per community_weighting.md §2–3:
        W_final = raw_weight
                  × exp(-alpha * |log(w_src / w_dst + ε)|)   [monetary factor]
                  × exp(-beta  * avg_gap / delta_w)           [temporal factor]

    Only called when cfg is passed to build_window_graph().
    Raw amounts are the default when cfg is None.

    Parameters
    ----------
    edges : pd.DataFrame
        Snapshot edge table with weight_col and optionally avg_gap_col.
    cfg : CommunityConfig
        alpha and beta are read from here.
    delta_w : int
        Window size used for temporal decay normalisation.
    avg_gap_col : str
        Column containing mean step gap per edge (from build_second_order_edges).
        If absent, temporal factor is 1.0 for all edges.

    Returns
    -------
    pd.DataFrame with weight_col replaced by the adjusted weight.
    Edges with weight below cfg.weight_filter_thresh are dropped.

    Notes
    -----
    - Direction is preserved — weights are applied symmetrically to amounts
      but the (src, dst) direction is never reversed.
    - A copy of the weight column is made; other columns are not touched.
    - Vectorized with numpy; no loops.
    """
    # w = edges[weight_col].to_numpy().astype(np.float64)
    # Compute per-node out/in totals from the edge table itself
    node_out = edges.groupby(src_col)[weight_col].sum().rename("node_out")
    node_in  = edges.groupby(dst_col)[weight_col].sum().rename("node_in")
    edges = edges.join(node_out, on=src_col).join(node_in, on=dst_col)
    w = edges[weight_col].to_numpy().astype(np.float64)
    money_factor = np.exp(
        -cfg.alpha * np.abs(
            np.log((edges["node_out"].to_numpy() / (edges["node_in"].to_numpy() + 1e-9)) + 1e-9)
        )
    )

    # Temporal decay factor
    if avg_gap_col in edges.columns:
        gaps = edges[avg_gap_col].to_numpy().astype(np.float64)
        time_factor = np.exp(-cfg.beta * gaps / max(delta_w, 1))
    else:
        time_factor = np.ones(len(edges), dtype=np.float64)

    w_final = (w * money_factor * time_factor).astype(np.float32)

    result = edges.copy()
    result[weight_col] = w_final

    # Drop weak edges
    result = result[result[weight_col] >= cfg.weight_filter_thresh].reset_index(drop=True)

    # Sparsification: keep top_k_neighbors per src node
    if cfg.top_k_neighbors > 0 and len(result) > 0:
        result = (
            result.sort_values([src_col, weight_col], ascending=[True, False])
            .groupby(src_col, as_index=False)
            .head(cfg.top_k_neighbors)
            .reset_index(drop=True)
        )

    return result


# ---------------------------------------------------------------------------
# Symmetrization fallback
# ---------------------------------------------------------------------------

def symmetrize(A: csr_matrix) -> csr_matrix:
    """
    Produce an undirected version of A: (A + A^T) / 2.

    Use only when the chosen algorithm (e.g. some Leiden variants) requires
    an undirected graph.  Document any call to this function — direction is
    lost and cannot be recovered from the result.

    Parameters
    ----------
    A : csr_matrix
        Directed weighted adjacency (output of build_window_graph).

    Returns
    -------
    csr_matrix — symmetrised, same shape as A.
    """
    return ((A + A.T) / 2.0).tocsr()


# ---------------------------------------------------------------------------
# Degree utilities (reused by detection and scoring)
# ---------------------------------------------------------------------------

def compute_degrees(A: csr_matrix) -> tuple[np.ndarray, np.ndarray, float]:
    """
    Compute weighted out-degree, in-degree, and total flow from A.

    Parameters
    ----------
    A : csr_matrix of shape (n, n)

    Returns
    -------
    (d_out, d_in, m_t)
        d_out : ndarray (n,) — row sums
        d_in  : ndarray (n,) — column sums
        m_t   : float — total weight
    """
    d_out = np.asarray(A.sum(axis=1)).flatten().astype(np.float32)
    d_in  = np.asarray(A.sum(axis=0)).flatten().astype(np.float32)
    m_t   = float(A.data.sum()) if A.nnz > 0 else 0.0
    return d_out, d_in, m_t


__all__ = [
    "WindowGraph",
    "build_window_graph",
    "apply_weighting",
    "symmetrize",
    "compute_degrees",
]

In [ ]:
"""
 End-to-end community detection orchestrator.

Execution order (community_pipeline.md §1):
    1. For each time window:
       a. Build directed weighted WindowGraph (from weighting.py)
       b. Run community detection (detection.py)
       c. Split oversized communities (detection.py)
       d. Track communities across windows (tracking.py)
       e. Extract community features (scoring.py)
       f. Release large objects immediately
    2. After all windows:
       a. Score all accumulated feature rows
       b. Build community assignment table
       c. Build suspicious community shortlist

Output tables:
    assignment_df : node-level table  [window_id, step_start, step_end,
                                       node, global_cid, community_size,
                                       overlap_score]
    feature_df    : community-level table with all scoring columns
    shortlist_df  : top-K suspicious communities

Memory rules (pipeline.md §4):
    - Window graph (WindowGraph) is built, used, and released per window.
    - No snapshot dict list accumulates over time.
    - Tracking buffer is capped at cfg.tracking_memory frames.
    - Feature rows are appended to a list; concatenated once at the end.

Guarantees:
    - Time      : windows processed in step_start order.
    - Direction : WindowGraph.A is directed; never symmetrised here.
    - Memory    : large objects deleted + gc.collect() after each window.
"""

from __future__ import annotations

import gc
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd



# ---------------------------------------------------------------------------
# Main pipeline function
# ---------------------------------------------------------------------------

def run_community_pipeline(
    window_iter,
    cfg: CommunityConfig,
    n_nodes: int,
    motif_counts_by_window: Optional[Dict[int, Dict[int, float]]] = None,
    n_alert: int = 0,
    n_rows: int = 1,
    src_col: str = "src_node",
    dst_col: str = "dst_node",
    amount_col: str = "amount",
    alert_col: str = "is_sar",
    weight_col: str = "amount",
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Run the end-to-end community detection pipeline.

    Parameters
    ----------
    window_iter : iterable of (step_start, step_end, window_df)
        Produces one window at a time. Matches iter_windows() from loader.py.
        window_df must have src_col, dst_col, amount_col.
    cfg : CommunityConfig
        All detection, tracking, and scoring hyperparameters.
    n_nodes : int
        Total number of encoded nodes (from NodeEncoder.n_nodes).
        Keeps adjacency matrix shape consistent across windows.
    motif_counts_by_window : Dict[window_id, Dict[global_cid, float]], optional
        Motif enrichment counts per community per window.
        Produced by motif module integration (features.py). If None, motif
        enrichment is 0 in the suspicion score.
    n_alert : int
        Total SAR-labelled rows in the full dataset (for AER reporting).
    n_rows : int
        Total rows in the full dataset (for AER reporting).
    src_col, dst_col, amount_col, alert_col, weight_col : str
        Column names in window DataFrames.
    verbose : bool
        Print per-window progress.

    Returns
    -------
    assignment_df : pd.DataFrame
        Node-level tracking table — one row per (window, node).
        Columns: window_id, step_start, step_end, node, global_cid,
                 community_size, overlap_score.

    feature_df : pd.DataFrame
        Community-level feature + score table — one row per (window, community).
        All fields from extract_community_features() + score_communities().

    shortlist_df : pd.DataFrame
        Top-K suspicious communities, filtered from feature_df.
    """
    global_cid_counter  = 0
    tracking_buffer:  List[pd.DataFrame] = []
    persistence_map:  Dict[int, int]     = {}   # global_cid → n_windows_seen
    assignment_rows:  List[pd.DataFrame] = []
    feature_rows:     List[pd.DataFrame] = []
    window_id         = 0
    total_windows     = 0

    for step_start, step_end, window_df in window_iter:
        total_windows += 1

        if len(window_df) < 2:   # skip windows with too few transactions
            window_id += 1
            continue

        if verbose:
            print(f"  Window {window_id}: steps [{step_start}, {step_end}] "
                  f"({len(window_df):,} rows)")

        # Guard: fail clearly if the weight column is missing
        # (build_window_graph would silently produce an all-zero adjacency).
        if weight_col not in window_df.columns:
            raise ValueError(
                f"weight_col='{weight_col}' not found in window_df. "
                f"Available columns: {list(window_df.columns)}"
            )

        # ── 1. Build directed weighted graph ──────────────────────────────────
        wg = build_window_graph(
            window_df,
            n_nodes=n_nodes,
            cfg=None,           # raw amount weights (no decay unless cfg passed)
            step_start=step_start,
            step_end=step_end,
            src_col=src_col,
            dst_col=dst_col,
            weight_col=weight_col,
        )

        if wg.m_t == 0:
            window_id += 1
            continue

        # ── 2. Community detection ─────────────────────────────────────────────
        labels = detect_communities(wg, cfg)
        if not labels:
            del wg
            gc.collect()
            window_id += 1
            continue

        # ── 3. Split oversized communities ────────────────────────────────────
        if cfg.s_max > 0:
            labels = split_large_communities(labels, wg, cfg)

        # ── 4. Cross-window tracking ──────────────────────────────────────────
        global_labels, global_cid_counter = match_communities_jaccard(
            labels, tracking_buffer, cfg, global_cid_counter
        )

        # ── 5. Build tracking record ──────────────────────────────────────────
        record = build_tracking_record(
            global_labels, window_id, step_start, step_end,
            prev_buffer=tracking_buffer,
        )
        tracking_buffer = update_buffer(tracking_buffer, record, cfg)

        # Update persistence counters
        for gcid in set(global_labels.values()):
            if gcid != -1:
                persistence_map[gcid] = persistence_map.get(gcid, 0) + 1

        assignment_rows.append(record)

        # ── 6. Node roles ─────────────────────────────────────────────────────
        node_roles = compute_node_roles(
            window_df, cfg,
            src_col=src_col, dst_col=dst_col,
            amount_col=amount_col, alert_col=alert_col,
        )

        # ── 7. Extract community features ────────────────────────────────────
        motif_counts = (
            motif_counts_by_window.get(window_id) if motif_counts_by_window else None
        )
        feat = extract_community_features(
            window_df=window_df,
            global_labels=global_labels,
            wg=wg,
            node_roles=node_roles,
            motif_counts=motif_counts,
            window_id=window_id,
            step_start=step_start,
            step_end=step_end,
            src_col=src_col,
            dst_col=dst_col,
            amount_col=amount_col,
            alert_col=alert_col,
        )

        if len(feat) > 0:
            feature_rows.append(feat)

        # ── 8. Release large objects ──────────────────────────────────────────
        del wg, labels, global_labels, node_roles, feat, record
        gc.collect()
        window_id += 1

    if verbose:
        print(f"\n  Processed {total_windows} windows, {global_cid_counter} global communities created.")

    # ── Assemble assignment table ─────────────────────────────────────────────
    if assignment_rows:
        assignment_df = pd.concat(assignment_rows, ignore_index=True)
    else:
        assignment_df = pd.DataFrame(columns=[
            "window_id", "step_start", "step_end", "node",
            "global_cid", "community_size", "overlap_score",
        ])
    del assignment_rows
    gc.collect()

    # ── Score all features ────────────────────────────────────────────────────
    if feature_rows:
        feature_df = pd.concat(feature_rows, ignore_index=True)
        del feature_rows
        gc.collect()

        feature_df = score_communities(
            feature_df,
            cfg,
            persistence_map=persistence_map,
            max_windows=max(total_windows, 1),
            n_alert=n_alert,
            n_rows=n_rows,
        )
    else:
        feature_df = pd.DataFrame()
        del feature_rows

    # ── Shortlist ─────────────────────────────────────────────────────────────
    if len(feature_df) > 0:
        shortlist_df = get_shortlist(feature_df, cfg)
    else:
        shortlist_df = pd.DataFrame()

    if verbose:
        print(f"  Shortlist: {len(shortlist_df)} suspicious communities (top-{cfg.top_k_export})")

    return assignment_df, feature_df, shortlist_df


# ---------------------------------------------------------------------------
# Export helper
# ---------------------------------------------------------------------------

def save_pipeline_outputs(
    assignment_df: pd.DataFrame,
    feature_df: pd.DataFrame,
    shortlist_df: pd.DataFrame,
    export_dir: str = "outputs/community",
    fmt: str = "parquet",
) -> None:
    """
    Save the three pipeline output tables to disk.

    Parameters
    ----------
    assignment_df, feature_df, shortlist_df : pd.DataFrame
    export_dir : str
        Target directory (set to "/content/drive/MyDrive/..." in Colab).
    fmt : str
        "parquet" (default) or "csv".

    Notes
    -----
    Directory is created if it does not exist.
    Parquet is preferred for downstream model consumption.
    """
    from pathlib import Path
    out = Path(export_dir)
    out.mkdir(parents=True, exist_ok=True)

    ext = "parquet" if fmt == "parquet" else "csv"
    tables = {
        "community_assignments": assignment_df,
        "community_features":    feature_df,
        "community_shortlist":   shortlist_df,
    }
    for name, df in tables.items():
        if len(df) == 0:
            continue
        path = out / f"{name}.{ext}"
        if fmt == "parquet":
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)
        print(f"  Saved {len(df):,} rows → {path}")


__all__ = [
    "run_community_pipeline",
    "save_pipeline_outputs",
]